## **Multimodal AI — Blood Work Analysis**

In [5]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

True

## Encode the image and send to the vision model

In [2]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

image_b64[:300]

'iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8eK+KYlVEqdvFBQtq64obVvsgUhatD9XSihuLFatYZRNUtA0iYF0AZZEiEBICSeb3ejy/O9+52QiRzfp5/5WcOXPmzDnDfJiZkzkaOI5jAAAAAOg6ihrrAAAAAACCKAAAAKA+uBIFAAAA1ARBFAAAAFATBFEAAABATRBEAQAAADVBEAUAAADUBEEUAAAAUBMEUQAAAEBN'

In [3]:
llm = ChatGroq(model="qwen/qwen3.8-27b")

message = HumanMessage(content=[
    {"type":"image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

Here is the extracted data from the blood work report, with abnormal values flagged based on the provided normal ranges.

### **Patient Information**
- **Name:** Rajesh Sharma
- **Age:** 48
- **Sex:** Male
- **Date:** May 7, 2026

---

### **Test Results & Abnormalities**

#### **COMPLETE BLOOD COUNT (CBC)**
| Test | Result | Normal Range | Status |
| :--- | :--- | :--- | :--- |
| Hemoglobin | 15.1 g/dL | 13.5–17.5 g/dL | Normal |
| Hematocrit | 44% | 41–53% | Normal |
| WBC | 6.8 x 10³/uL | 4.5–11.0 x 10³/uL | Normal |
| Platelets | 220 x 10³/uL | 150–400 x 10³/uL | Normal |

#### **LIPID PANEL**
| Test | Result | Normal Range | Status |
| :--- | :--- | :--- | :--- |
| Total Cholesterol | 238 mg/dL | < 200 mg/dL | ⚠️ **HIGH** |
| LDL Cholesterol | 162 mg/dL | < 100 mg/dL | ⚠️ **HIGH** |
| HDL Cholesterol | 36 mg/dL | > 40 mg/dL | ⚠️ **LOW** |
| Triglycerides | 188 mg/dL | < 150 mg/dL | ⚠️ **HIGH** |

#### **METABOLIC PANEL**
| Test | Result | Normal Range | Status |
| :--- | :--- | :-

## Suggest Diet Plan Agent

The agent reads the blood work image, categorises the condition, then calls the diet tool.

In [6]:
@tool
def get_diet_recommendation(condition: str) -> str:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [7]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [9]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a good diet plan."},
    ])]
})

print(result["messages"][-1].content)

Here is the analysis of the blood work report for **Rajesh Sharma (Age 48, Male)** dated May 7, 2026:

### 1. Data Extraction

**COMPLETE BLOOD COUNT (CBC)**
*   **Hemoglobin:** 15.1 g/dL (Normal: 13.5–17.5) – *Normal*
*   **Hematocrit:** 44% (Normal: 41–53%) – *Normal*
*   **WBC:** 6.8 x 10^3/uL (Normal: 4.5–11.0) – *Normal*
*   **Platelets:** 220 x 10^3/uL (Normal: 150–400) – *Normal*

**LIPID PANEL**
*   **Total Cholesterol:** 238 mg/dL (Normal: <200) – **High**
*   **LDL Cholesterol:** 162 mg/dL (Normal: <100) – **High**
*   **HDL Cholesterol:** 36 mg/dL (Normal: >40) – **Low**
*   **Triglycerides:** 188 mg/dL (Normal: <150) – **High**

**METABOLIC PANEL**
*   **Glucose (Fasting):** 92 mg/dL (Normal: 70–99) – *Normal*
*   **HbA1c:** 5.3% (Normal: <5.7%) – *Normal*
*   **Creatinine:** 1.0 mg/dL (Normal: 0.7–1.3) – *Normal*
*   **eGFR:** 82 mL/min (Normal: >60) – *Normal*

### 2. Condition Categorization
Based on the lipid panel results showing total cholesterol, LDL, and triglycerid